In [1]:
import pygadm
import pandas as pd
import geopandas as gpd
import pycountry
from pathlib import Path
import os

from bokeh.plotting import output_notebook, output_file
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10

output_notebook()

Loading BokehJS ...

In [2]:
df = pd.read_parquet("../../results/country_confustion_matrix.parquet")

In [3]:
df["f1"]  = 2 * df.TP / (2 * df.TP + df.FP + df.FN)

In [4]:
import pycountry_convert as pc

def country_to_continent(country_name):
    try:
        country_code = pc.country_name_to_country_alpha2(country_name)
        continent_code = pc.country_alpha2_to_continent_code(country_code)
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
    except Exception as e:
        return None  # or handle the exception/log as needed

In [5]:
df["continent"] = df["country"].apply(country_to_continent)
df = df[~df.continent.isna()]

In [6]:
hdi_df = pd.read_csv("../../data/human_development_index.csv")[["iso3", "hdi_2020"]]

In [7]:
df = pd.merge(left=df, right=hdi_df, how="left", left_on="gid", right_on="iso3")
df = df[~df.hdi_2020.isna()].rename(columns={"hdi_2020": "hdi"}).drop(columns="iso3")

In [8]:
df

,country,gid,pixel_count,TN,FP,FN,TP,f1,continent,hdi
1,Georgia,GEO,438373,385107,7893,28983,16390,0.470598,Asia,0.802
2,Gabon,GAB,1238379,1226274,5402,4557,2146,0.301172,Africa,0.710
3,"Micronesia, Federated States of",FSM,3592,3166,13,337,76,0.302789,Oceania,0.629
5,France,FRA,3715986,2825570,165799,427637,296980,0.500221,Europe,0.898
7,Finland,FIN,3624346,3498970,73358,16569,35449,0.440839,Europe,0.938
8,Ethiopia,ETH,5347815,5119052,2073,218575,8115,0.068516,Africa,0.498
9,Estonia,EST,405438,383109,3532,12895,5902,0.418122,Europe,0.892
10,Spain,ESP,3086931,2679750,145495,76600,185086,0.625009,Europe,0.899
11,Eritrea,ERI,584115,576126,92,7520,377,0.090127,Africa,0.494
12,Egypt,EGY,5138834,4855242,120655,14312,148625,0.687733,Africa,0.734


In [10]:
continents = df['continent'].unique().tolist()
palette = Category10[max(3, len(continents))]

source = ColumnDataSource.from_df(df)


fig = figure(
    x_axis_label='Human Development Index', 
    y_axis_label='F1',
    width=980
)
fig.scatter(
    x="hdi",
    y="f1", 
    source=source,
    size=10, 
    alpha=0.9,
    color=factor_cmap('continent', palette=palette, factors=continents),
    legend_field='continent',
)

hover = HoverTool(tooltips=[
    ("Country", "@country"),
    ("F1", "@f1"),
    ("Pixel Count", "@pixel_count"),
])

fig.legend.location = "bottom_right"

fig.add_tools(hover)
show(fig)